# TNT V2: boundary- and word-aware Gold-train ablation

This notebook develops the proposed TNT improvement **without using Gold-dev or Gold-test**. It creates a video-disjoint inner split from Gold-train and compares four controlled variants. The V2 namespace fixes the KEEP-dominant operation loss, unused-segment PAD dominance, raw-reference validation, and slow GPU reconstruction observed in the first pilot:

1. character heads only;
2. character + explicit spacing boundary head;
3. character + word-level KEEP/EDIT gate;
4. the full boundary- and word-aware hybrid.

Each training cell is independent. Set only the corresponding `RUN_...` switch to `True`. Results are experimental and do not replace the frozen M7 benchmark.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    candidates = [parent for parent in PROJECT_ROOT.parents if (parent / 'src').is_dir()]
    if not candidates:
        raise FileNotFoundError('Run this notebook from the PFE repository or one of its subdirectories.')
    PROJECT_ROOT = candidates[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import torch
from IPython.display import display

from src.modeling.tnt_hybrid_experiment import (
    ABLATIONS,
    RESULTS_DIR,
    HybridExperimentProtocol,
    ablation_summary,
    auxiliary_label_profile,
    prepare_inner_gold_split,
    prediction_comparison,
    run_ablation,
)

print('Project:', PROJECT_ROOT)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

c:\Users\thioy\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Project: C:\Users\thioy\Desktop\School\PFE
PyTorch: 2.8.0+cu129
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## Protocol

The four variants use the same encoder dimensions, random seed, optimizer, epoch budget and inner split. Only the auxiliary loss and inference switches change. The initial thresholds are architecture-motivated defaults, not values selected from Gold-dev/test. Edit-class weights use one fixed, bounded inverse-square-root formula computed from inner-training labels only; this keeps SUBSTITUTE/DELETE/EXPAND and rare spacing actions from disappearing under their majority classes.

In [2]:
PROTOCOL = HybridExperimentProtocol(
    seed=2026,
    epochs=10,
    learning_rate=3e-4,
    train_batch_size=32,
    evaluation_batch_size=24,
    output_slots_per_source=16,
    boundary_loss_weight=0.5,
    word_loss_weight=0.5,
    boundary_confidence_threshold=0.60,
    word_keep_threshold=0.80,
    balance_edit_classes=True,
    maximum_class_weight=4.0,
    generated_eval_interval_epochs=1,
)
print(PROTOCOL)
print('Ablations:', ABLATIONS)
print('Results:', RESULTS_DIR.resolve())

HybridExperimentProtocol(seed=2026, epochs=10, learning_rate=0.0003, train_batch_size=32, evaluation_batch_size=24, output_slots_per_source=32, boundary_loss_weight=0.5, word_loss_weight=0.5, boundary_confidence_threshold=0.6, word_keep_threshold=0.8, balance_edit_classes=True, maximum_class_weight=4.0, generated_eval_interval_epochs=1)
Ablations: ('character_only', 'boundary_only', 'word_only', 'full_hybrid')
Results: C:\Users\thioy\Desktop\School\PFE\results\m7_seq2seq\tnt_hybrid_gold_train_ablation_v2


## Gold-train inner split

One complete video is held out. This prevents near-duplicate style from the same video appearing on both sides of the architecture-development split.

In [3]:
inner_train, inner_validation, split_manifest = prepare_inner_gold_split(PROTOCOL)
display(pd.DataFrame([split_manifest]).T.rename(columns={0: 'value'}))
label_profile = auxiliary_label_profile(inner_train, PROTOCOL)
display(pd.DataFrame({
    'count': {
        **{f'operation_{key}': value for key, value in label_profile['operation_counts'].items()},
        **{f'boundary_{key}': value for key, value in label_profile['boundary_counts'].items()},
        **{f'word_{key}': value for key, value in label_profile['word_counts'].items()},
    },
}))
print('Operation weights:', label_profile['operation_class_weights'])
print('Boundary weights:', label_profile['boundary_class_weights'])
print('Word weights:', label_profile['word_class_weights'])
display(inner_train[['source_id', 'source_text', 'target_text']].head(3))
display(inner_validation[['source_id', 'source_text', 'target_text']].head(3))

,value
created_at_utc,2026-09-28T19:40:24.165941+00:00
experiment,tnt_hybrid_gold_train_ablation_v2
source,C:\Users\thioy\Desktop\School\PFE\data\annotat...
source_sha256,5ef420b2d2b4fd29d643fb7d6c1c0bb368b7f6f186c23a...
policy,video-disjoint inner split made only from lock...
selection_rule,video size closest to 20% of Gold-train; seede...
seed,2026
held_out_video,https://www.youtube.com/watch?v=8CAy6LdHR4M
training_rows,118
validation_rows,24


,count
operation_keep,4978
operation_delete,383
operation_substitute,589
operation_expand,198
boundary_none,4929
boundary_keep_space,997
boundary_delete_space,10
boundary_insert_space_before,94
word_keep,579
word_edit,546


Operation weights: (0.5556604516836584, 2.003261048961719, 1.6153970533797293, 2.7861490022298274)
Boundary weights: (0.5530307135923884, 1.229648774376234, 4.0, 4.0)
Word weights: (0.9856483098314404, 1.0149974286764376)


,source_id,source_text,target_text
0,gold:train:250,mann buneh weurseuk rek sama hol bi jekh shete...,man bu ne wërsëg rek sama xol bi jeex séttétét
1,gold:train:253,tidiane aka wayadi thi dome,tidiane aka wayadi ci doom
2,gold:train:254,maman oumy mom bane épisode lay dèwè?,maman oumy moom ban épisode lay dee


,source_id,source_text,target_text
0,gold:train:3521,mais song bi kouko way li nekh na,mais song bi ku ko way li neex na
1,gold:train:3534,amdy ioe amdy ioe diémo dra,amdy yow amdy yow jéemo dara
2,gold:train:3541,taxi clando bi tmitt aussi sounou na,taxi clando bi tamit aussi sonn na


## A — Character-only control

Both new losses and both new inference mechanisms are disabled. This measures the shared V2 character backbone before attributing gains to either addition.

In [4]:
RUN_CHARACTER_ONLY = True
if RUN_CHARACTER_ONLY:
    character_only_result = run_ablation(
        'character_only', inner_train, inner_validation, PROTOCOL, resume=True
    )
    display(character_only_result.get('development_metrics', {}))
else:
    print('Skipped. Set RUN_CHARACTER_ONLY=True to train this condition.')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[TNT-V2] Created hybrid model with 1,742,282 parameters: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v2\initializations\character_only\tnt_hybrid_char_b795371e5fc3
[TNT-V2] character_only: 118 inner-train rows, 24 inner-validation rows
[M7] Loading C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v2\initializations\character_only\tnt_hybrid_char_b795371e5fc3


You are using a model of type `wolof_tnt_hybrid_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/46 [00:00<?, ?it/s]

[M7] Loaded 1,742,282 total parameters; training 1,742,282 (100.0000%) via full_finetune
[M7] Tokenizing 118 training and 24 development pairs


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v2\ablations\character_only\attempt_001\best_model\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v2\ablations\character_only\attempt_001\best_model\tokenizer_config.json
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v2\ablations\character_only\attempt_001\best_model\tokenizer_config.json


[M7] Running development prediction and final metrics


The following columns in the test set don't have a corresponding argument in `TntHybridTransformerForConditionalGeneration.forward` and have been ignored: target_text, source_text, source_id. If target_text, source_text, source_id are not expected by `TntHybridTransformerForConditionalGeneration.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 24
  Batch size = 24


{'rows': 24,
 'cer': 0.2113564668769716,
 'wer': 0.5169082125603864,
 'chrf': 60.99868471546182,
 'exact_match': 0.0,
 'correction_tp': 0,
 'correction_fp': 4,
 'correction_fn': 52,
 'correction_precision': 0.0,
 'correction_recall': 0.0,
 'correction_f1': 0.0,
 'overcorrection_rate': 1.0,
 'unchanged_reference_rows': 0,
 'unchanged_sentence_overcorrection_rate': 0.0}

## B — Boundary head only

Adds direct supervision for keeping, deleting and inserting spaces. Pure spacing errors do not mark a word as lexically incorrect.

In [5]:
RUN_BOUNDARY_ONLY = True
if RUN_BOUNDARY_ONLY:
    boundary_only_result = run_ablation(
        'boundary_only', inner_train, inner_validation, PROTOCOL, resume=True
    )
    display(boundary_only_result.get('development_metrics', {}))
else:
    print('Skipped. Set RUN_BOUNDARY_ONLY=True to train this condition.')

Generate config GenerationConfig {
  "bos_token_id": 1,
  "decoder_start_token_id": 1,
  "eos_token_id": 2,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 0
}

Configuration saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\initializations\boundary_only\tnt_hybrid_char_279524784ef3\config.json
Configuration saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\initializations\boundary_only\tnt_hybrid_char_279524784ef3\generation_config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\initializations\boundary_only\tnt_hybrid_char_279524784ef3\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\initializations\boundary_only\tnt_hybrid_char_279524784ef3\tokenizer_config.json


[TNT-V2] Created hybrid model with 1,742,282 parameters: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\initializations\boundary_only\tnt_hybrid_char_279524784ef3
[TNT-V2] boundary_only: 118 inner-train rows, 24 inner-validation rows
[M7] Loading C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\initializations\boundary_only\tnt_hybrid_char_279524784ef3


loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\initializations\boundary_only\tnt_hybrid_char_279524784ef3\config.json
loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\initializations\boundary_only\tnt_hybrid_char_279524784ef3\config.json
You are using a model of type `wolof_tnt_hybrid_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Model config PreTrainedConfig {
  "architectures": [
    "TntHybridTransformerForConditionalGeneration"
  ],
  "attention_heads": 4,
  "bos_token_id": 1,
  "boundary_class_weights": [
    0.5530307135923884,
    1.229648774376234,
  

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\ablations\boundary_only\attempt_001\best_model\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\ablations\boundary_only\attempt_001\best_model\tokenizer_config.json
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\ablations\boundary_only\attempt_001\best_model\tokenizer_config.json


[M7] Running development prediction and final metrics


The following columns in the test set don't have a corresponding argument in `TntHybridTransformerForConditionalGeneration.forward` and have been ignored: target_text, source_id, source_text. If target_text, source_id, source_text are not expected by `TntHybridTransformerForConditionalGeneration.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 24
  Batch size = 24


{'rows': 24,
 'cer': 0.21052631578947367,
 'wer': 0.5120772946859904,
 'chrf': 61.43143753015135,
 'exact_match': 0.0,
 'correction_tp': 1,
 'correction_fp': 3,
 'correction_fn': 52,
 'correction_precision': 0.25,
 'correction_recall': 0.018867924528301886,
 'correction_f1': 0.03508771929824561,
 'overcorrection_rate': 0.75,
 'unchanged_reference_rows': 0,
 'unchanged_sentence_overcorrection_rate': 0.0}

## C — Word KEEP/EDIT gate only

Pools the characters of each whitespace-delimited source token and predicts whether the token should be preserved or delegated to the character edit heads. It does not yet generate a lexical candidate.

In [ ]:
RUN_WORD_ONLY = True
if RUN_WORD_ONLY:
    word_only_result = run_ablation(
        'word_only', inner_train, inner_validation, PROTOCOL, resume=True
    )
    display(word_only_result.get('development_metrics', {}))
else:
    print('Skipped. Set RUN_WORD_ONLY=True to train this condition.')

Generate config GenerationConfig {
  "bos_token_id": 1,
  "decoder_start_token_id": 1,
  "eos_token_id": 2,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 0
}

Configuration saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\initializations\word_only\tnt_hybrid_char_f68afc8bd6b3\config.json
Configuration saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\initializations\word_only\tnt_hybrid_char_f68afc8bd6b3\generation_config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\initializations\word_only\tnt_hybrid_char_f68afc8bd6b3\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\initializations\word_only\tnt_hybrid_char_f68afc8bd6b3\tokenizer_config.json


[TNT-V2] Created hybrid model with 1,742,282 parameters: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\initializations\word_only\tnt_hybrid_char_f68afc8bd6b3
[TNT-V2] word_only: 118 inner-train rows, 24 inner-validation rows
[M7] Loading C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\initializations\word_only\tnt_hybrid_char_f68afc8bd6b3


loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\initializations\word_only\tnt_hybrid_char_f68afc8bd6b3\config.json
loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\initializations\word_only\tnt_hybrid_char_f68afc8bd6b3\config.json
You are using a model of type `wolof_tnt_hybrid_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Model config PreTrainedConfig {
  "architectures": [
    "TntHybridTransformerForConditionalGeneration"
  ],
  "attention_heads": 4,
  "bos_token_id": 1,
  "boundary_class_weights": [
    0.5530307135923884,
    1.229648774376234,
    4.0,
 

...

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\ablations\word_only\attempt_001\checkpoint-12\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\ablations\word_only\attempt_001\checkpoint-12\tokenizer_config.json
The following columns in the Evaluation set don't have a corresponding argument in `TntHybridTransformerForConditionalGeneration.forward` and have been ignored: target_text, source_id, source_text. If target_text, source_id, source_text are not expected by `TntHybridTransformerForConditionalGeneration.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 24
  Batch size = 24
Saving model checkpoint to C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\ablations\word_only\attempt_001\checkpoint-16
Configuration saved in C:\Users\thioy\Desktop\Sc

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\ablations\word_only\attempt_001\checkpoint-16\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\ablations\word_only\attempt_001\checkpoint-16\tokenizer_config.json
The following columns in the Evaluation set don't have a corresponding argument in `TntHybridTransformerForConditionalGeneration.forward` and have been ignored: target_text, source_id, source_text. If target_text, source_id, source_text are not expected by `TntHybridTransformerForConditionalGeneration.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 24
  Batch size = 24
Saving model checkpoint to C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\ablations\word_only\attempt_001\checkpoint-20
Configuration saved in C:\Users\thioy\Desktop\Sc

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\ablations\word_only\attempt_001\checkpoint-20\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_gold_train_ablation_v1\ablations\word_only\attempt_001\checkpoint-20\tokenizer_config.json


## D — Full hybrid

Trains and applies both additions. A gain over the two single-addition variants would indicate complementary information; otherwise the simpler successful component should be preferred.

In [ ]:
RUN_FULL_HYBRID = True
if RUN_FULL_HYBRID:
    full_hybrid_result = run_ablation(
        'full_hybrid', inner_train, inner_validation, PROTOCOL, resume=True
    )
    display(full_hybrid_result.get('development_metrics', {}))
else:
    print('Skipped. Set RUN_FULL_HYBRID=True to train this condition.')

## Compare completed runs

Interpret CER, WER and chrF together, then inspect correction precision/recall and overcorrection. The experiment is useful only if it improves word/boundary behavior without merely editing more text.

In [ ]:
summary = ablation_summary()
if summary.empty:
    print('No completed ablation yet.')
else:
    preferred = [
        'ablation', 'cer', 'wer', 'chrf', 'exact_match',
        'correction_precision', 'correction_recall', 'correction_f1',
        'overcorrection_rate', 'attempt',
    ]
    display(summary[[column for column in preferred if column in summary.columns]])

## Qualitative comparison

Read these rows before selecting an architecture. In particular, check whether lower WER comes from correct boundary/word decisions or simply from more aggressive editing.

In [ ]:
predictions = prediction_comparison()
if predictions.empty:
    print('No completed predictions yet.')
else:
    display(predictions)